# Robot cell: FANUC CR-35iA and UR20 pass an apple

Two FANUC CR-35iA and two UR20 stand around one table. Every robot carries an OnRobot RG6 gripper.
The robots pass an apple around the table: `cr35_1` takes it at the west edge and leaves it at the
south edge, `ur20_1` carries it to the east edge, `ur20_2` to the north edge and `cr35_2` back to the
west edge. One round takes 36.8 s.

- Robots: `fanuc_cr35ia_support` (ROS-Industrial), `ur_description` (Universal Robots)
- Gripper: `onrobot_rg_description` (UOsaka-Harada-Laboratory/onrobot, MIT)
- Layout: `ros/biba_cell_description/config/cell_layout.yaml`
- Motion and apple spots: `ros/biba_cell_description/config/motions.yaml`

## Start the cell

Run the cell below, then click **Start cell**. RVizWeb opens on the right with a view of the whole cell.
Without *animate*, every robot holds its first waypoint. *speed* scales the motion.

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from lab_launch import LaunchPanel

pkg = Path.home() / "work/ros/biba_cell_description"
animate = widgets.Checkbox(value=True, description="animate")
speed = widgets.FloatSlider(value=1.0, min=0.25, max=2.0, step=0.25, description="speed")

cell = LaunchPanel(
    "cell",
    lambda: ["biba_cell_description", "cell.launch",
             f"animate:={str(animate.value).lower()}", f"speed:={speed.value}"],
    rvizweb_config=pkg / "rviz/cell.json",
)
display(widgets.HBox([animate, speed]))
cell.show()

## Layout and apple spots

The waypoints were computed with inverse kinematics for exactly this layout
(`compute_relay.py` in the planning folder). A robot moved relative to the table misses the apple
spots, so the waypoints have to be recomputed after changing the layout.

In [ ]:
import yaml

layout = yaml.safe_load((pkg / "config/cell_layout.yaml").read_text())
motions = yaml.safe_load((pkg / "config/motions.yaml").read_text())
for r in layout["robots"]:
    print(f"{r['id']:8s} {r['type']:7s} x={r['x']:5.2f} y={r['y']:5.2f} yaw={r['yaw']:5.2f} "
          f"pedestal={r.get('pedestal', 0):.2f} gripper={r.get('gripper', '-')}")
for t in layout.get("tables", []):
    print(f"{t['id']:8s} table   x={t['x']:5.2f} y={t['y']:5.2f} size={t['size']}")
for name, p in motions["apple"]["spots"].items():
    print(f"apple spot {name:6s} {p}")

## What is running

`robot_state_publisher` computes the transforms, including the mimic joints of the gripper fingers.
`waypoint_player` publishes the joint positions on `/joint_states` and the apple on `/cell/objects`.

In [ ]:
!rosnode list
!timeout 4 rostopic hz /joint_states
!timeout 4 rostopic hz /cell/objects